In [ ]:
# =============================================================================
# 1. IMPORTS & CONFIGURATION
# =============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px  # Optionnel, pour des graphes interactifs si besoin

# Configuration du style visuel "Pro"
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Palette de couleurs personnalisée (Cybersecurity style: Rouge, Orange, Gris)
CYBER_PALETTE = sns.color_palette("rocket", as_cmap=False)

print("✅ Bibliothèques chargées et configuration appliquée.")

# =============================================================================
# 2. CHARGEMENT ET NETTOYAGE DES DONNÉES
# =============================================================================
# On charge le CSV généré par le script précédent
try:
    df = pd.read_csv("anssi_cve_dataframe.csv")
    print(f"📂 Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes.")
except FileNotFoundError:
    print("❌ ERREUR : Le fichier 'anssi_cve_dataframe.csv' est introuvable.")
    # Création d'un dataset dummy pour l'exemple si le fichier manque
    print("⚠️ Génération de données factices pour la démonstration...")
    data = {
        'Score CVSS': np.random.uniform(2, 10, 100),
        'Score EPSS': np.random.exponential(0.1, 100),
        'Type CWE': np.random.choice(['CWE-79', 'CWE-89', 'CWE-20', 'CWE-787', 'Autres'], 100),
        'Éditeur/Vendor': np.random.choice(['Microsoft', 'Cisco', 'Adobe', 'Google', 'Linux'], 100),
        'Date de publication': pd.date_range(start='2023-01-01', periods=100),
        'Type de bulletin': np.random.choice(['Alerte', 'Avis'], 100)
    }
    df = pd.DataFrame(data)

# --- NETTOYAGE ---
# 1. Conversion des scores en numérique (gestion des "Non disponible")
df['Score CVSS'] = pd.to_numeric(df['Score CVSS'], errors='coerce')
df['Score EPSS'] = pd.to_numeric(df['Score EPSS'], errors='coerce')

# 2. Conversion des dates
df['Date de publication'] = pd.to_datetime(df['Date de publication'], utc=True, errors='coerce')

# 3. Remplissage des valeurs manquantes pour les catégories
df['Type CWE'] = df['Type CWE'].fillna("Inconnu")
df['Éditeur/Vendor'] = df['Éditeur/Vendor'].fillna("Inconnu")

# 4. Création d'une catégorie de gravité CVSS
def classer_gravite(score):
    if pd.isna(score): return "Inconnu"
    if score >= 9.0: return "CRITIQUE"
    if score >= 7.0: return "ÉLEVÉE"
    if score >= 4.0: return "MOYENNE"
    return "FAIBLE"

df['Gravité'] = df['Score CVSS'].apply(classer_gravite)

# Aperçu après nettoyage
print("✅ Nettoyage terminé.")
display(df.head())
display(df.info())

# =============================================================================
# 3. ANALYSE DES SCORES CVSS (GRAVITÉ)
# =============================================================================
plt.figure(figsize=(10, 6))

# Histogramme avec KDE (Kernel Density Estimate)
sns.histplot(data=df, x="Score CVSS", kde=True, bins=20, color="firebrick", alpha=0.6)

plt.title("Distribution des Scores CVSS (Gravité des vulnérabilités)", fontsize=16, fontweight='bold')
plt.xlabel("Score CVSS (0-10)")
plt.ylabel("Nombre de CVE")
plt.axvline(x=7.0, color='orange', linestyle='--', label='Seuil Élevé (7.0)')
plt.axvline(x=9.0, color='red', linestyle='--', label='Seuil Critique (9.0)')
plt.legend()
plt.show()

# Camembert des niveaux de gravité
plt.figure(figsize=(8, 8))
gravite_counts = df['Gravité'].value_counts()
colors = {'CRITIQUE': '#d62728', 'ÉLEVÉE': '#ff7f0e', 'MOYENNE': '#ffbb78', 'FAIBLE': '#2ca02c', 'Inconnu': '#7f7f7f'}

plt.pie(gravite_counts, labels=gravite_counts.index, autopct='%1.1f%%', startangle=140, 
        colors=[colors.get(x, '#999') for x in gravite_counts.index], wedgeprops={'edgecolor': 'white'})
plt.title("Répartition par Niveau de Gravité", fontsize=16)
plt.show()

# =============================================================================
# 4. ANALYSE DES TYPES DE FAIBLESSES (CWE)
# =============================================================================
plt.figure(figsize=(12, 6))

# On garde le top 10 des CWE, on regroupe le reste en "Autres"
top_n = 10
cwe_counts = df['Type CWE'].value_counts()
if len(cwe_counts) > top_n:
    top_cwe = cwe_counts[:top_n]
    others = pd.Series([cwe_counts[top_n:].sum()], index=['Autres'])
    cwe_final = pd.concat([top_cwe, others])
else:
    cwe_final = cwe_counts

sns.barplot(x=cwe_final.values, y=cwe_final.index, palette="viridis")
plt.title(f"Top {top_n} des Types de Vulnérabilités (CWE)", fontsize=16, fontweight='bold')
plt.xlabel("Nombre de CVE détectées")
plt.ylabel("Identifiant CWE")
plt.show()

# =============================================================================
# 5. ANALYSE CROISÉE : CVSS vs EPSS (Le Risque Réel)
# =============================================================================
# L'EPSS est souvent très bas (<0.1). On utilise une échelle log ou on zoom.

plt.figure(figsize=(10, 8))

# Scatter plot
scatterplot = sns.scatterplot(
    data=df, 
    x="Score CVSS", 
    y="Score EPSS", 
    hue="Gravité", 
    size="Score EPSS", 
    sizes=(20, 200),
    palette=colors,
    alpha=0.7
)

plt.title("Corrélation : Gravité (CVSS) vs Probabilité d'Exploitation (EPSS)", fontsize=16, fontweight='bold')
plt.xlabel("Score CVSS (Théorique)")
plt.ylabel("Score EPSS (Probabilité réelle d'attaque)")
plt.axhline(y=0.1, color='purple', linestyle=':', label='Seuil critique EPSS (ex: 10%)')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.) # Légende à l'extérieur
plt.tight_layout()
plt.show()

# Heatmap de corrélation
plt.figure(figsize=(6, 5))
corr_matrix = df[['Score CVSS', 'Score EPSS']].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matrice de Corrélation (CVSS vs EPSS)")
plt.show()

# =============================================================================
# 6. FOCUS ÉDITEURS & PRODUITS
# =============================================================================
# Top 10 Éditeurs touchés
top_vendors = df['Éditeur/Vendor'].value_counts().head(10).index

plt.figure(figsize=(12, 6))
df_top_vendors = df[df['Éditeur/Vendor'].isin(top_vendors)]

# Boxplot pour voir la dispersion des scores par éditeur
sns.boxplot(data=df_top_vendors, x='Éditeur/Vendor', y='Score CVSS', palette="mako")
plt.xticks(rotation=45)
plt.title("Dispersion des scores CVSS pour les 10 éditeurs les plus affectés", fontsize=16)
plt.show()

# Analyse type de bulletin par éditeur
plt.figure(figsize=(12, 6))
sns.countplot(data=df_top_vendors, x='Éditeur/Vendor', hue='Type de bulletin', palette="Set2")
plt.xticks(rotation=45)
plt.title("Répartition Avis vs Alertes pour les Top Éditeurs", fontsize=16)
plt.legend(title='Type de Bulletin')
plt.show()

# =============================================================================
# 7. ÉVOLUTION TEMPORELLE
# =============================================================================
plt.figure(figsize=(14, 6))

# On groupe par jour
df_time = df.set_index('Date de publication').sort_index()
# Resample par jour et cumulatif
df_cumul = df_time.resample('D').size().cumsum()

sns.lineplot(data=df_cumul, linewidth=3, color="#2c3e50")
plt.fill_between(df_cumul.index, df_cumul.values, color="#2c3e50", alpha=0.1)

plt.title("Évolution cumulative des CVE détectées dans le temps", fontsize=16, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Nombre total de CVE cumulées")
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()

# =============================================================================
# 8. CONCLUSION & PRIORISATION (Pour l'étape suivante)
# =============================================================================
# Extraction des "Must Fix Now" : CVSS > 9 ET EPSS > 0.01 (par exemple)
must_fix = df[ (df['Score CVSS'] >= 9.0) & (df['Score EPSS'] > 0.01) ]

print("="*60)
print("🚀 RÉSULTAT DE L'ANALYSE POUR LE MODULE D'ALERTE")
print("="*60)
print(f"Nombre total de vulnérabilités analysées : {len(df)}")
print(f"Nombre de vulnérabilités CRITIQUES (CVSS >= 9) : {len(df[df['Score CVSS']>=9])}")
print(f"Nombre de vulnérabilités à RISQUE RÉEL IMMÉDIAT (CVSS>=9 + EPSS élevé) : {len(must_fix)}")
print("\n🔍 Top 3 des vulnérabilités les plus dangereuses (à alerter en priorité) :")
display(must_fix[['Identifiant CVE', 'Score CVSS', 'Score EPSS', 'Éditeur/Vendor', 'Produit']].head(3))

ModuleNotFoundError: No module named 'plotly'